# Modelos CART

Se evalúa una configuración base de árboles CART para clasificación y regresión. La intención es disponer de una referencia sencilla antes de comparar métodos de Bagging y Boosting.


## Librerías


In [ ]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    recall_score,
    r2_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, TimeSeriesSplit, cross_val_score, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor


## Carga de datos procesados


In [ ]:
bank_train_con_duration = pd.read_csv("data/processed/serie_computacional/bank_train_con_duration.csv")
bank_test_con_duration = pd.read_csv("data/processed/serie_computacional/bank_test_con_duration.csv")
bank_train_sin_duration = pd.read_csv("data/processed/serie_computacional/bank_train_sin_duration.csv")
bank_test_sin_duration = pd.read_csv("data/processed/serie_computacional/bank_test_sin_duration.csv")

consumo_train = pd.read_csv("data/processed/serie_computacional/power_train_hourly.csv")
consumo_test = pd.read_csv("data/processed/serie_computacional/power_test_hourly.csv")

resumen_datos = pd.DataFrame([
    {"Dataset": "Bank Marketing", "Escenario": "con duration", "Entrenamiento": len(bank_train_con_duration), "Prueba": len(bank_test_con_duration), "Variables": bank_train_con_duration.shape[1] - 1},
    {"Dataset": "Bank Marketing", "Escenario": "sin duration", "Entrenamiento": len(bank_train_sin_duration), "Prueba": len(bank_test_sin_duration), "Variables": bank_train_sin_duration.shape[1] - 1},
    {"Dataset": "Electric Power", "Escenario": "regresión horaria", "Entrenamiento": len(consumo_train), "Prueba": len(consumo_test), "Variables": consumo_train.shape[1] - 2},
])

resumen_datos


## Funciones auxiliares


In [ ]:
def preparar_preprocesador(X):
    variables_numericas = [col for col in X.columns if pd.api.types.is_numeric_dtype(X[col])]
    variables_categoricas = [col for col in X.columns if col not in variables_numericas]

    try:
        codificador = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        codificador = OneHotEncoder(handle_unknown="ignore", sparse=False)

    pasos = []
    if variables_numericas:
        pasos.append(("numericas", SimpleImputer(strategy="median"), variables_numericas))
    if variables_categoricas:
        pasos.append(("categoricas", codificador, variables_categoricas))

    return ColumnTransformer(pasos), variables_numericas, variables_categoricas


def separar_bank(entrenamiento, prueba):
    X_entrenamiento = entrenamiento.drop(columns=["y"])
    y_entrenamiento = entrenamiento["y"]
    X_prueba = prueba.drop(columns=["y"])
    y_prueba = prueba["y"]
    return X_entrenamiento, y_entrenamiento, X_prueba, y_prueba


def separar_consumo(entrenamiento, prueba):
    objetivo = "Global_active_power"
    columnas_excluidas = [objetivo, "datetime"]
    X_entrenamiento = entrenamiento.drop(columns=columnas_excluidas)
    y_entrenamiento = entrenamiento[objetivo]
    X_prueba = prueba.drop(columns=columnas_excluidas)
    y_prueba = prueba[objetivo]
    return X_entrenamiento, y_entrenamiento, X_prueba, y_prueba


def evaluar_clasificacion(modelo, nombre, X_entrenamiento, y_entrenamiento, X_prueba, y_prueba, escenario):
    preprocesador, numericas, categoricas = preparar_preprocesador(X_entrenamiento)
    flujo = Pipeline([("preprocesamiento", preprocesador), ("modelo", modelo)])
    inicio = time.perf_counter()
    flujo.fit(X_entrenamiento, y_entrenamiento)
    tiempo_ajuste = time.perf_counter() - inicio
    predicciones = flujo.predict(X_prueba)
    probabilidades = flujo.predict_proba(X_prueba)[:, 1] if hasattr(flujo, "predict_proba") else None

    return flujo, {
        "Dataset": "Bank Marketing",
        "Escenario": escenario,
        "Modelo": nombre,
        "F1 de la clase positiva": f1_score(y_prueba, predicciones, pos_label=1, zero_division=0),
        "Precisión": precision_score(y_prueba, predicciones, pos_label=1, zero_division=0),
        "Sensibilidad": recall_score(y_prueba, predicciones, pos_label=1, zero_division=0),
        "AUC-ROC": roc_auc_score(y_prueba, probabilidades) if probabilidades is not None else np.nan,
        "Exactitud": accuracy_score(y_prueba, predicciones),
        "Tiempo de ajuste (s)": tiempo_ajuste,
        "Variables originales": X_entrenamiento.shape[1],
        "Variables numéricas": len(numericas),
        "Variables categóricas": len(categoricas),
    }


def evaluar_regresion(modelo, nombre, X_entrenamiento, y_entrenamiento, X_prueba, y_prueba):
    preprocesador, numericas, categoricas = preparar_preprocesador(X_entrenamiento)
    flujo = Pipeline([("preprocesamiento", preprocesador), ("modelo", modelo)])
    inicio = time.perf_counter()
    flujo.fit(X_entrenamiento, y_entrenamiento)
    tiempo_ajuste = time.perf_counter() - inicio
    predicciones = flujo.predict(X_prueba)
    mse = mean_squared_error(y_prueba, predicciones)

    return flujo, {
        "Dataset": "Individual Household Electric Power Consumption",
        "Modelo": nombre,
        "RMSE": np.sqrt(mse),
        "MAE": mean_absolute_error(y_prueba, predicciones),
        "MSE": mse,
        "R²": r2_score(y_prueba, predicciones),
        "Tiempo de ajuste (s)": tiempo_ajuste,
        "Variables originales": X_entrenamiento.shape[1],
        "Variables numéricas": len(numericas),
        "Variables categóricas": len(categoricas),
    }


## Configuración base de CART

Se utiliza una configuración controlada para limitar la complejidad del árbol y hacer comparable la primera referencia. `max_depth` limita la profundidad, `min_samples_leaf` evita hojas con muy pocas observaciones y `class_weight="balanced"` compensa el desequilibrio de clases en Bank Marketing. No se interpreta como una selección óptima de hiperparámetros.


In [ ]:
filas_clasificacion = []

for escenario, entrenamiento, prueba in [
    ("con duration", bank_train_con_duration, bank_test_con_duration),
    ("sin duration", bank_train_sin_duration, bank_test_sin_duration),
]:
    X_entrenamiento, y_entrenamiento, X_prueba, y_prueba = separar_bank(entrenamiento, prueba)
    modelo = DecisionTreeClassifier(
        max_depth=5,
        min_samples_leaf=50,
        class_weight="balanced",
        random_state=42,
    )
    _, fila = evaluar_clasificacion(
        modelo, "CART", X_entrenamiento, y_entrenamiento, X_prueba, y_prueba, escenario
    )
    filas_clasificacion.append(fila)

resultados_clasificacion = pd.DataFrame(filas_clasificacion)
resultados_clasificacion


## CART en regresión

La regresión eléctrica se mantiene como un problema contemporáneo: se usan mediciones del mismo intervalo horario y una división cronológica entre entrenamiento y prueba.


In [ ]:
X_entrenamiento, y_entrenamiento, X_prueba, y_prueba = separar_consumo(consumo_train, consumo_test)

modelo = DecisionTreeRegressor(
    max_depth=8,
    min_samples_leaf=50,
    random_state=42,
)

_, fila_regresion = evaluar_regresion(
    modelo, "CART", X_entrenamiento, y_entrenamiento, X_prueba, y_prueba
)

resultados_regresion = pd.DataFrame([fila_regresion])
resultados_regresion


## Guardado de resultados


In [ ]:
Path("reports/tables/serie_computacional").mkdir(parents=True, exist_ok=True)

resultados_clasificacion.to_csv(
    "reports/tables/serie_computacional/cart_clasificacion_metricas.csv",
    index=False,
)
resultados_regresion.to_csv(
    "reports/tables/serie_computacional/cart_regresion_metricas.csv",
    index=False,
)

pd.DataFrame([
    {"archivo": "cart_clasificacion_metricas.csv"},
    {"archivo": "cart_regresion_metricas.csv"},
])
